## NB_patient_360 (Silver to Gold)
* **Target:** `<catalog>.gold.patient_360`, one row per patient
* **Driving table:** `<catalog>.silver.patients`
* **Joins (LEFT):** `hospitals` on `registered_hospital_id`, `insurance_providers` on `insurance_provider_id`, aggregated `lab_results` by `patient_id`
* **Load type:** FULL rebuild with `CREATE OR REPLACE TABLE ... AS SELECT`

Implementation notes (based on the silver schema):
* `patients.date_of_birth` is stored as TIMESTAMP in silver, so it is cast to DATE.
* `insurance_providers` and `lab_results` contain duplicate keys from repeated loads. Each is deduplicated to the latest row per key so the join cannot multiply rows or double-count tests.

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
print(f"Building {target_table} from {source_schema}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurer and lab test summary'
AS
WITH insurance AS (
  SELECT *
  FROM {source_schema}.insurance_providers
  QUALIFY ROW_NUMBER() OVER (PARTITION BY insurance_provider_id
                             ORDER BY update_timestamp DESC, landing_timestamp DESC) = 1
),
lab AS (
  SELECT *
  FROM {source_schema}.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id
                             ORDER BY update_timestamp DESC, landing_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                               AS total_tests,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count,
    MAX(test_date)                         AS last_test_date
  FROM lab
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                   AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                        AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  i.provider_name                                                                 AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                         AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                                 AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                                 AS critical_test_count,
  l.last_test_date
FROM {source_schema}.patients p
LEFT JOIN {source_schema}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))